In [ ]:
!pip install -q pydantic>=2.7.0 pydantic-ai openai>=1.37.0 python-dotenv rich duckduckgo-search

### Build a complete AI agent 

In this tutorial, we'll build a complete AI agent from scratch. We'll start with 
the basics and progressively add more sophisticated capabilities.

Think of this as building a digital assistant that can actually DO things,
not just chat. We'll create an agent that can research, analyze, and make decisions.

Learning Objectives:
- Set up a PydanticAI agent from scratch
- Define clear agent goals and capabilities  
- Implement custom tools for your agent
- Test and validate agent behavior


In [ ]:
import os
from typing import List, Optional, Dict, Any
from datetime import datetime
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
import requests
import json
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

### Defining Our Agent's Purpose

Before we write any code, we need to be crystal clear about what our agent should do.
Let's create a research assistant that can find information and provide analysis.


In [ ]:
class ResearchQuery(BaseModel):
    """
    This defines what kind of research requests our agent can handle.
    Being specific here helps our agent understand its boundaries.
    """
    topic: str = Field(description="The main topic to research")
    specific_questions: List[str] = Field(description="Specific questions to answer about the topic")
    research_depth: str = Field(description="Either 'surface', 'detailed', or 'comprehensive'")
    target_audience: str = Field(description="Who is this research for? e.g., 'students', 'professionals', 'general public'")

In [ ]:
class ResearchResult(BaseModel):
    """
    Simple research result structure.
    """
    topic: str = Field(description="The researched topic")
    summary: str = Field(description="High-level summary of findings")
    key_findings: List[str] = Field(description="Most important discoveries")


### Creating the Basic Agent Structure

Now let's create our agent with a clear system prompt that defines its personality and approach.


In [ ]:
research_agent = Agent(
    openrouter_model,
    output_type=ResearchResult,
    system_prompt="""
    You are an expert research assistant with a methodical, analytical approach.
    
    Your strengths:
    - Breaking down complex topics into understandable parts
    - Finding reliable, current information
    - Providing balanced, objective analysis
    - Tailoring explanations to your audience
    
    Your process:
    1. Understand what the user really wants to know
    2. Gather information from multiple angles
    3. Synthesize findings into clear, actionable insights
    4. Always be honest about limitations and uncertainty
    
    Remember: Quality over quantity. Better to provide fewer, well-researched 
    insights than many shallow ones.
    """
)

### Adding Information Gathering Tools

Tools are what make agents actually useful. Let's start with a simple web search tool.


In [ ]:
@research_agent.tool
async def search_web(ctx: RunContext[None], query: str, num_results: int = 3) -> str:
    """
    Search the web for current information about a topic.
    This is our agent's window to the current world.
    """
    # In a real implementation, you'd use a search API like Serper, Tavily, or Google Custom Search
    # For this demo, we'll simulate search results
    
    simulated_results = {
        "artificial intelligence": [
            "AI refers to computer systems that can perform tasks typically requiring human intelligence",
            "Current AI applications include natural language processing, computer vision, and robotics",
            "Key challenges include ethics, bias, and ensuring AI alignment with human values"
        ],
        "climate change": [
            "Global temperatures have risen by approximately 1.1°C since pre-industrial times",
            "Main causes include greenhouse gas emissions from fossil fuels and deforestation", 
            "Solutions include renewable energy transition and carbon capture technologies"
        ],
        "quantum computing": [
            "Quantum computers use quantum mechanical phenomena to process information",
            "They have potential to solve certain problems exponentially faster than classical computers",
            "Current challenges include quantum error correction and maintaining quantum coherence"
        ]
    }
    
    # Find the best match for the query
    best_match = None
    for topic, results in simulated_results.items():
        if topic.lower() in query.lower() or query.lower() in topic.lower():
            best_match = results
            break
    
    if best_match is None:
        best_match = ["Limited information available for this specific query"]
    
    return f"Search results for '{query}':\n" + "\n".join([f"- {result}" for result in best_match[:num_results]])


### Adding Synthesis Capabilities

Our agent needs to combine information from multiple sources into coherent insights.


In [ ]:
@research_agent.tool
async def synthesize_information(ctx: RunContext[None], information_pieces: List[str]) -> str:
    """
    Combine multiple pieces of information into coherent insights.
    This is where our agent shows its analytical intelligence.
    """
    if not information_pieces:
        return "No information provided for synthesis"
    
    # Basic synthesis logic (in practice, this would be more sophisticated)
    synthesis = f"""
    Information Synthesis Summary:
    
    Total sources analyzed: {len(information_pieces)}
    
    Common themes identified:
    - {len([piece for piece in information_pieces if 'technology' in piece.lower()])} sources mention technology aspects
    - {len([piece for piece in information_pieces if any(word in piece.lower() for word in ['challenge', 'problem', 'issue'])])} sources discuss challenges
    - {len([piece for piece in information_pieces if any(word in piece.lower() for word in ['solution', 'approach', 'method'])])} sources offer solutions
    
    Synthesis approach: Cross-referencing information for consistency and identifying consensus views.
    """
    
    return synthesis


### Let's run it!

In [ ]:
# Define our research request
research_request = "I need to understand artificial intelligence for a college presentation. What are the key concepts, current applications, and main challenges?"

print(f"Research Request: {research_request}\n")
print("🤖 Agent is working...\n")

# Run the agent
result = await research_agent.run(research_request)

# Display the structured results
print("="*60)
print("RESEARCH RESULTS")
print("="*60)
print(f"Topic: {result.output.topic}")
print()

print("📋 SUMMARY:")
print(result.output.summary)
print()

print("🔍 KEY FINDINGS:")
for i, finding in enumerate(result.output.key_findings, 1):
    print(f"{i}. {finding}")
print()